# Orquestador principal (orquestador de orquestadores)
Generado con la skill `notebooks-medallion` desde `metadata/config.json`. **No lo edites a mano.**

Ejecuta en serie, y se para en el primer fallo:
1. `orq_bronze_silver`
2. `orq_silver_gold`
3. Refresco de los modelos semánticos: `GreatAdventures/GreatAdventuresVentas`.

Cada orquestador hijo escribe su fila en `log_orquestacion`; cada refresco de modelo, una fila en `log_ejecucion` (capa `modelo_semantico`).
Este notebook escribe una fila en `log_orquestacion` con capa `principal` y los `id_orquestacion` de los hijos en `parametros`.

In [ ]:
proyecto = "GreatAdventures"
ws_control = "GreatAdventuresControl"
lh_control = "control"
simular = False            # se pasa a los orquestadores hijos; con True tampoco se refrescan los modelos
modo = "auto"              # auto | reconstruir, se pasa a orq_bronze_silver
solo_capas = ""            # p. ej. "orq_silver_gold" para saltarse bronze → silver; vacío = todas
procesar_modelos = True    # False: no refresca los modelos semánticos
timeout_orquestador_s = 43200
timeout_refresco_s = 3600

In [ ]:
# Utilidades comunes (iguales en todos los notebooks generados): rutas, log en el lakehouse de control y MERGE.
import json, uuid, datetime as dt
from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T, Window

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def ruta_lakehouse(ws, lh):
    return f"abfss://{ws}@{ONELAKE}/{lh}.Lakehouse"

RUTA_CONTROL = f"{ruta_lakehouse(ws_control, lh_control)}/Tables/dbo"
ID_EJECUCION = str(uuid.uuid4())
INICIO = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

def contexto(clave, defecto=None):
    try:
        return notebookutils.runtime.context.get(clave) or defecto
    except Exception:
        return defecto

ESQUEMA_LOG_EJECUCION = T.StructType([
    T.StructField("id_ejecucion", T.StringType()),
    T.StructField("id_orquestacion", T.StringType()),
    T.StructField("proyecto", T.StringType()),
    T.StructField("capa", T.StringType()),
    T.StructField("objeto", T.StringType()),
    T.StructField("destino", T.StringType()),
    T.StructField("tipo_carga", T.StringType()),
    T.StructField("estado", T.StringType()),
    T.StructField("simulado", T.BooleanType()),
    T.StructField("inicio", T.TimestampType()),
    T.StructField("fin", T.TimestampType()),
    T.StructField("duracion_s", T.DoubleType()),
    T.StructField("ficheros_leidos", T.LongType()),
    T.StructField("filas_leidas", T.LongType()),
    T.StructField("filas_unicas", T.LongType()),
    T.StructField("filas_duplicadas", T.LongType()),
    T.StructField("filas_borrado_origen", T.LongType()),
    T.StructField("filas_insertadas", T.LongType()),
    T.StructField("filas_actualizadas", T.LongType()),
    T.StructField("filas_borradas", T.LongType()),
    T.StructField("marca_desde", T.StringType()),
    T.StructField("marca_hasta", T.StringType()),
    T.StructField("error", T.StringType()),
    T.StructField("notebook", T.StringType()),
    T.StructField("workspace", T.StringType()),
    T.StructField("usuario", T.StringType()),
    T.StructField("spark_app_id", T.StringType()),
    T.StructField("parametros", T.StringType()),
])

def registrar_ejecucion(**valores):
    """Añade una fila a control.dbo.log_ejecucion. Solo append: no hay conflictos entre notebooks en paralelo."""
    fin = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    base = dict(id_ejecucion=ID_EJECUCION, proyecto=proyecto, inicio=INICIO, fin=fin,
                duracion_s=(fin - INICIO).total_seconds(), notebook=contexto("currentNotebookName"),
                workspace=contexto("currentWorkspaceName"), usuario=contexto("userName"),
                spark_app_id=spark.sparkContext.applicationId)
    base.update(valores)
    fila = tuple(base.get(c.name) for c in ESQUEMA_LOG_EJECUCION)
    (spark.createDataFrame([fila], ESQUEMA_LOG_EJECUCION)
         .write.format("delta").mode("append").option("mergeSchema", "true")
         .save(f"{RUTA_CONTROL}/log_ejecucion"))

def es_tabla_delta(ruta):
    try:
        return DeltaTable.isDeltaTable(spark, ruta)
    except Exception:
        return False

def con_hash(df, excluir):
    """Añade _hash_fila: sha2 de las columnas de negocio. Sirve para no reescribir filas que no han cambiado."""
    cols = [c for c in df.columns if c not in excluir and not c.startswith("_")]
    return df.withColumn("_hash_fila", F.sha2(F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0000")) for c in sorted(cols)]), 256))

def anadir_columnas_nuevas(ruta, df, evolucion):
    """Evolución de esquema controlada: añade al destino las columnas nuevas del origen o falla si no está permitido."""
    destino = spark.read.format("delta").load(ruta)
    nuevas = [f for f in df.schema.fields if f.name not in destino.columns]
    if not nuevas:
        return []
    if not evolucion:
        raise ValueError(f"Columnas nuevas en origen y evolucion_esquema=False: {[f.name for f in nuevas]}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in nuevas)
    spark.sql(f"ALTER TABLE delta.`{ruta}` ADD COLUMNS ({cols})")
    return [f.name for f in nuevas]

def columnas_con_tipo_cambiado(ruta, df):
    """Columnas que existen en destino con un tipo distinto al del origen."""
    actual = {f.name: f.dataType for f in spark.read.format("delta").load(ruta).schema.fields}
    return [f"{f.name}: {actual[f.name].simpleString()} -> {f.dataType.simpleString()}"
            for f in df.schema.fields if f.name in actual and actual[f.name] != f.dataType]

def fusionar(ruta, df, claves, borrar_ausentes, col_borrado=None, cambio_tipo="ignorar"):
    """MERGE contra Delta. df ya viene deduplicado por claves y con _hash_fila.
    - col_borrado (opcional): columna booleana del origen; si es True, la fila se borra en destino.
    - borrar_ausentes: borra del destino lo que no viene en el origen (instantánea completa).
    - cambio_tipo: qué hacer si una columna existente cambia de tipo en el origen.
        "ignorar"    el MERGE convierte al tipo del destino (comportamiento histórico).
        "reescribir" reescribe la tabla con el esquema del origen (solo tiene sentido con borrar_ausentes).
        "fallar"     lanza un error.
    Devuelve (insertadas, actualizadas, borradas)."""
    cols = [c for c in df.columns if c != col_borrado]
    if not es_tabla_delta(ruta):
        datos = df.filter(~F.col(col_borrado)) if col_borrado else df
        datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
        return spark.read.format("delta").load(ruta).count(), 0, 0
    if cambio_tipo != "ignorar":
        cambiados = columnas_con_tipo_cambiado(ruta, df.select(*cols))
        if cambiados and cambio_tipo == "fallar":
            raise ValueError(f"Cambio de tipo en {cambiados} y la tabla no se puede reescribir (borrar_ausentes=False)")
        if cambiados:
            print(f"Cambio de tipo en {cambiados}: se reescribe la tabla con el esquema de la consulta")
            antes = spark.read.format("delta").load(ruta).count()
            datos = df.filter(~F.col(col_borrado)) if col_borrado else df
            datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
            return spark.read.format("delta").load(ruta).count(), 0, antes
    anadir_columnas_nuevas(ruta, df.select(*cols), evolucion_esquema)
    cond = " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in claves)
    asignar = {f"`{c}`": f"s.`{c}`" for c in cols}
    m = DeltaTable.forPath(spark, ruta).alias("t").merge(df.alias("s"), cond)
    if col_borrado:
        m = m.whenMatchedDelete(condition=f"s.`{col_borrado}`")
        m = m.whenMatchedUpdate(condition=f"NOT s.`{col_borrado}` AND NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(condition=f"NOT s.`{col_borrado}`", values=asignar)
    else:
        m = m.whenMatchedUpdate(condition="NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(values=asignar)
    if borrar_ausentes:
        m = m.whenNotMatchedBySourceDelete()
    m.execute()
    met = DeltaTable.forPath(spark, ruta).history(1).select("operationMetrics").first()[0] or {}
    return (int(met.get("numTargetRowsInserted", 0)), int(met.get("numTargetRowsUpdated", 0)),
            int(met.get("numTargetRowsDeleted", 0)))

In [ ]:
import time, requests

CAPA = "principal"
ID_ORQUESTACION = ID_EJECUCION
ORQUESTADORES = ["orq_bronze_silver", "orq_silver_gold"]
MODELOS = [
    {
        "modelo": "GreatAdventuresVentas",
        "workspace": "GreatAdventures"
    }
]
API_PBI = "https://api.powerbi.com/v1.0/myorg"

def cabeceras():
    return {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}", "Content-Type": "application/json"}

def id_por_nombre(url, nombre, campo="name"):
    r = requests.get(url, headers=cabeceras(), timeout=60)
    r.raise_for_status()
    ids = [x["id"] for x in r.json()["value"] if x.get(campo) == nombre]
    if not ids:
        raise ValueError(f"No encuentro '{nombre}' en {url}")
    return ids[0]

def refrescar_modelo(workspace, modelo):
    """Refresco completo con la API de refresh mejorada de Power BI; espera a que termine."""
    ws = id_por_nombre(f"{API_PBI}/groups?$filter=name eq '{workspace}'", workspace)
    ds = id_por_nombre(f"{API_PBI}/groups/{ws}/datasets", modelo)
    r = requests.post(f"{API_PBI}/groups/{ws}/datasets/{ds}/refreshes", headers=cabeceras(), timeout=60,
                      json={"type": "full", "commitMode": "transactional", "retryCount": 1})
    if r.status_code != 202:
        raise RuntimeError(f"Refresco de {modelo} rechazado: {r.status_code} {r.text[:500]}")
    url_estado = r.headers.get("Location") or f"{API_PBI}/groups/{ws}/datasets/{ds}/refreshes/{r.headers['RequestId']}"
    limite = time.time() + timeout_refresco_s
    while time.time() < limite:
        time.sleep(15)
        e = requests.get(url_estado, headers=cabeceras(), timeout=60).json()
        estado = e.get("extendedStatus") or e.get("status")
        if estado in ("Completed", "Failed", "Cancelled", "Disabled", "TimedOut"):
            if estado != "Completed":
                detalle = json.dumps(e.get("messages") or e.get("objects") or e, ensure_ascii=False)[:3000]
                raise RuntimeError(f"Refresco de {modelo}: {estado}. {detalle}")
            return
    raise TimeoutError(f"Refresco de {modelo} sin terminar tras {timeout_refresco_s} s")

In [ ]:
estado, error, hijos = "OK", None, {}
capas = [o for o in ORQUESTADORES if not solo_capas.strip() or o in [s.strip() for s in solo_capas.split(",")]]
for orq in capas:
    args = dict(simular=simular)
    if orq == "orq_bronze_silver":
        args["modo"] = modo
    print(f"→ {orq}")
    try:
        salida = notebookutils.notebook.run(orq, timeout_orquestador_s, args)
        hijos[orq] = json.loads(salida) if salida else {}
    except Exception as e:
        estado, error = "ERROR", f"{orq}: {type(e).__name__}: {e}"[:4000]
        hijos[orq] = {"estado": "ERROR"}
        break
    print(f"  {orq}: {hijos[orq].get('estado')} · id_orquestacion={hijos[orq].get('id_orquestacion')}")

# Modelos semánticos: solo si las capas han ido bien y no es una simulación.
modelos_ok = modelos_error = 0
if estado == "OK" and procesar_modelos and not simular:
    for m in MODELOS:
        inicio = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
        est, err = "OK", None
        print(f"→ refresco {m['workspace']}/{m['modelo']}")
        try:
            refrescar_modelo(m["workspace"], m["modelo"])
        except Exception as e:
            est, err = "ERROR", f"{type(e).__name__}: {e}"[:4000]
        fin = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
        registrar_ejecucion(id_orquestacion=ID_ORQUESTACION, capa="modelo_semantico", objeto=m["modelo"],
                            destino=f"{m['workspace']}/{m['modelo']}", tipo_carga="refresh", estado=est,
                            simulado=False, error=err, inicio=inicio, fin=fin,
                            duracion_s=(fin - inicio).total_seconds(), parametros=json.dumps(m, ensure_ascii=False))
        print(f"  {m['modelo']}: {est}" + (f" · {err}" if err else ""))
        if est == "OK":
            modelos_ok += 1
        else:
            modelos_error += 1
    if modelos_error:
        estado, error = "ERROR", f"{modelos_error} modelos semánticos con error. Consulta log_ejecucion con id_orquestacion={ID_ORQUESTACION}"

In [ ]:
ok = sum(1 for h in hijos.values() if h.get("estado") == "OK") + modelos_ok
errores = sum(1 for h in hijos.values() if h.get("estado") == "ERROR") + modelos_error
actividades = len(capas) + (len(MODELOS) if procesar_modelos and not simular else 0)
sin_log = actividades - ok - errores   # lo que no llegó a ejecutarse por un fallo anterior
fin = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
esquema = T.StructType([T.StructField(n, t) for n, t in [
    ("id_orquestacion", T.StringType()), ("proyecto", T.StringType()), ("capa", T.StringType()),
    ("estado", T.StringType()), ("inicio", T.TimestampType()), ("fin", T.TimestampType()),
    ("duracion_s", T.DoubleType()), ("actividades", T.LongType()), ("ok", T.LongType()),
    ("sin_cambios", T.LongType()), ("errores", T.LongType()), ("sin_log", T.LongType()),
    ("concurrencia", T.LongType()), ("parametros", T.StringType()), ("error", T.StringType()),
    ("notebook", T.StringType()), ("usuario", T.StringType()), ("spark_app_id", T.StringType())]])
parametros = dict(simular=simular, modo=modo, solo_capas=solo_capas, procesar_modelos=procesar_modelos,
                  hijos={k: v.get("id_orquestacion") for k, v in hijos.items()})
fila = (ID_ORQUESTACION, proyecto, CAPA, estado, INICIO, fin, (fin - INICIO).total_seconds(), actividades,
        ok, 0, errores, sin_log, 1, json.dumps(parametros, ensure_ascii=False), error,
        contexto("currentNotebookName"), contexto("userName"), spark.sparkContext.applicationId)
(spark.createDataFrame([fila], esquema).write.format("delta").mode("append").option("mergeSchema", "true")
     .save(f"{RUTA_CONTROL}/log_orquestacion"))
print(f"{CAPA}: {estado} · {ok} OK, {errores} con error, {sin_log} sin ejecutar · id_orquestacion={ID_ORQUESTACION}")
if estado == "ERROR":
    raise RuntimeError(error)

In [ ]:
notebookutils.notebook.exit(json.dumps(dict(estado=estado, id_orquestacion=ID_ORQUESTACION, ok=ok, errores=errores,
                                            hijos={k: v.get("id_orquestacion") for k, v in hijos.items()})))